In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import OrdinalEncoder
import os
import pandas as pd
from scipy.spatial.distance import cdist # For de-embedding reference

In [14]:
# Load the data
x_cat_train = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/X_cat_train.npy', allow_pickle=True)
x_num_train = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/X_num_train.npy', allow_pickle=True)
y_train = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/y_train.npy', allow_pickle=True)

x_cat_test = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/X_cat_test.npy', allow_pickle=True)
x_num_test = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/X_num_test.npy', allow_pickle=True)
y_test = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/y_test.npy', allow_pickle=True)

x_cat_val = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/X_cat_val.npy', allow_pickle=True)
x_num_val = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/X_num_val.npy', allow_pickle=True)
y_val = np.load('/home/fjunpop/tab-ddpm-dp/data/churn2/y_val.npy', allow_pickle=True)

# Get unique values

# 1. Unique values across all categorical features in the training set
print("--- Unique values in x_cat_train (across all columns) ---")
unique_x_cat_train = np.unique(x_cat_train)
print(unique_x_cat_train)

print("\n--- Unique values per categorical feature (column) in x_cat_train ---")
# Iterate through each column (feature) of the 2D array
for i in range(x_cat_train.shape[1]):
    # Slice the array to get all rows for the i-th column: x_cat_train[:, i]
    unique_col_i = np.unique(x_cat_train[:, i])
    print(f"Feature {i} (Cat_{i}): {len(unique_col_i)} unique IDs -> {unique_col_i}")

print("\n--- Unique values in y_train (class labels) ---")
# The target variable usually contains the class labels (e.g., [0, 1] for binary classification)
unique_y_train = np.unique(y_train)
print(unique_y_train)

print(x_num_train[:5])
print(x_cat_train[:5])

# print(x_cat_train.shape)
# print(x_num_train.shape)
# print(y_train.shape)

--- Unique values in x_cat_train (across all columns) ---
['0' '1' 'France' 'Germany' 'Spain']

--- Unique values per categorical feature (column) in x_cat_train ---
Feature 0 (Cat_0): 3 unique IDs -> ['France' 'Germany' 'Spain']
Feature 1 (Cat_1): 2 unique IDs -> ['0' '1']
Feature 2 (Cat_2): 2 unique IDs -> ['0' '1']
Feature 3 (Cat_3): 2 unique IDs -> ['0' '1']

--- Unique values in y_train (class labels) ---
[0 1]
[[8.3800000e+02 3.5000000e+01 0.0000000e+00 0.0000000e+00 2.0000000e+00
  1.9730591e+05 1.9730591e+05]
 [6.6100000e+02 4.4000000e+01 1.0000000e+00 1.4113662e+05 1.0000000e+00
  1.8974278e+05 1.8974278e+05]
 [5.4700000e+02 6.2000000e+01 1.0000000e+01 1.2773875e+05 2.0000000e+00
  8.5153000e+04 8.5153000e+04]
 [6.1500000e+02 4.1000000e+01 4.0000000e+00 1.3038582e+05 1.0000000e+00
  1.3066195e+05 1.3066195e+05]
 [7.2100000e+02 4.0000000e+01 3.0000000e+00 0.0000000e+00 1.0000000e+00
  1.4487467e+05 1.4487467e+05]]
[['Spain' '1' '0' '1']
 ['Germany' '1' '1' '0']
 ['France' '1' '

### Encoder V1.1

In [19]:
# Set a seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# --- Define Paths and Output Directory ---
# NOTE: The script requires the BASE_PATH files to exist
BASE_PATH = '/home/fjunpop/tab-ddpm-dp/data/churn2/'
OUTPUT_DIR = '/home/fjunpop/tab-ddpm-dp/data/churn2cont/'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------
# 0. Load Data using Specified Paths
# ---------------------------------------------

print("Loading data from specified paths...")

# Load training data
x_cat_train_np = np.load(os.path.join(BASE_PATH, 'X_cat_train.npy'), allow_pickle=True)
x_num_train_np = np.load(os.path.join(BASE_PATH, 'X_num_train.npy'), allow_pickle=True)
y_train_np = np.load(os.path.join(BASE_PATH, 'y_train.npy'), allow_pickle=True)

# Load testing data
x_cat_test_np = np.load(os.path.join(BASE_PATH, 'X_cat_test.npy'), allow_pickle=True)
x_num_test_np = np.load(os.path.join(BASE_PATH, 'X_num_test.npy'), allow_pickle=True)
y_test_np = np.load(os.path.join(BASE_PATH, 'y_test.npy'), allow_pickle=True)

# Load validation data
x_cat_val_np = np.load(os.path.join(BASE_PATH, 'X_cat_val.npy'), allow_pickle=True)
x_num_val_np = np.load(os.path.join(BASE_PATH, 'X_num_val.npy'), allow_pickle=True)
y_val_np = np.load(os.path.join(BASE_PATH, 'y_val.npy'), allow_pickle=True)


# --- Encoding and Type Conversion ---
print("Converting categorical strings to integer IDs...")

# 1. Handle TEXTUAL Column (Index 0: Country/Area)
x_country_all = np.vstack([x_cat_train_np[:, 0:1], x_cat_val_np[:, 0:1], x_cat_test_np[:, 0:1]])
country_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
country_encoder.fit(x_country_all)

# Transform the Country/Area column (output is float)
x_cat_train_np[:, 0] = country_encoder.transform(x_cat_train_np[:, 0:1]).flatten()
x_cat_val_np[:, 0] = country_encoder.transform(x_cat_val_np[:, 0:1]).flatten()
x_cat_test_np[:, 0] = country_encoder.transform(x_cat_test_np[:, 0:1]).flatten()


# 2. CONVERSION FIX: Convert all 4 categorical columns to integer IDs.
def safe_cat_to_int(arr):
    # Step 1: Convert to float first to handle any decimal strings
    arr_float = arr.astype(np.float64) 
    # Step 2: Convert to integer
    return arr_float.astype(np.int64)

x_cat_train_np = safe_cat_to_int(x_cat_train_np)
x_cat_val_np = safe_cat_to_int(x_cat_val_np)
x_cat_test_np = safe_cat_to_int(x_cat_test_np)


# 3. Ensure numerical and target data are float32
x_num_train_np = x_num_train_np.astype(np.float32)
x_num_val_np = x_num_val_np.astype(np.float32)
x_num_test_np = x_num_test_np.astype(np.float32)

y_train_np = y_train_np.astype(np.float32)
y_val_np = y_val_np.astype(np.float32)
y_test_np = y_test_np.astype(np.float32)

print("Conversion complete. Categorical data is ready for embedding. ✅")

# Convert NumPy arrays to PyTorch Tensors for training
x_cat_train = torch.from_numpy(x_cat_train_np).long()
x_num_train = torch.from_numpy(x_num_train_np).float()
y_train = torch.from_numpy(y_train_np).float()

# --- DEFINE MISSING VARIABLES ---
C_cat = x_cat_train_np.shape[1] 
C_num = x_num_train_np.shape[1]

# N_classes FIX: Determine the number of unique classes from the target array
unique_labels = np.unique(y_train_np)
N_classes = len(unique_labels)
# --------------------------------

# Determine category dimensions and set embedding sizes
cat_dims = [x_cat_train[:, i].max().item() + 1 for i in range(C_cat)]
embed_dims = [3] * C_cat 
total_emb_dim = sum(embed_dims)


# --- VISUALIZATION: BEFORE EMBEDDING ---
print("\n" + "="*90)
print("VISUALIZATION: X_train BEFORE EMBEDDING (First 5 Rows - Encoded Integers + Floats)")
df_cat = pd.DataFrame(x_cat_train_np[:5], columns=[f'Cat_{i}' for i in range(C_cat)])
df_num = pd.DataFrame(x_num_train_np[:5], columns=[f'Num_{i}' for i in range(C_num)])
df_mixed_before = pd.concat([df_num, df_cat], axis=1)

# NOTE: The expected total columns (11) is hardcoded and may be incorrect for your data.
print(df_mixed_before.to_string(float_format='%.4f'))
print(f"Total columns BEFORE embedding: {C_num} (Num) + {C_cat} (Cat) = {C_num + C_cat}") 
print("="*90 + "\n")


# ---------------------------------------------
# 1. Learn Embeddings (PyTorch Model)
# ---------------------------------------------

class EmbeddingNet(nn.Module):
    def __init__(self, cat_dims, embed_dims, num_features, n_classes):
        super().__init__()
        self.embedding_layers = nn.ModuleList([
            nn.Embedding(cat_dim, emb_dim) 
            for cat_dim, emb_dim in zip(cat_dims, embed_dims)
        ])
        
        self.fc = nn.Sequential(
            nn.Linear(num_features + sum(embed_dims), 32),
            nn.ReLU(),
            # Output dimension must match N_classes (e.g., 1 for binary classification with BCEWithLogitsLoss)
            nn.Linear(32, 1 if n_classes == 2 else n_classes) 
        )

    def forward(self, x_cat, x_num):
        embeddings = [layer(x_cat[:, i]) for i, layer in enumerate(self.embedding_layers)]
        # x_num is a 2D tensor (Batch_size, C_num)
        x_all = torch.cat([x_num] + embeddings, dim=1)
        return self.fc(x_all)

# Instantiate and train the model
# NOTE: Adjusted n_classes input to 1 if binary classification, since you are using BCEWithLogitsLoss.
model = EmbeddingNet(cat_dims, embed_dims, C_num, N_classes) 
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
epochs = 5
batch_size = 32 

# Create DataLoader for training
train_dataset = TensorDataset(x_cat_train, x_num_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print(f"Training model to learn embeddings (Epochs: {epochs}, Total Embedding Dim: {total_emb_dim})...")
model.train()
for epoch in range(epochs):
    for x_cat_batch, x_num_batch, y_batch in train_loader:
        optimizer.zero_grad()
        # outputs shape will be (batch_size, 1), y_batch shape is (batch_size)
        outputs = model(x_cat_batch, x_num_batch) 
        # Squeeze outputs to (batch_size) to match y_batch for BCEWithLogitsLoss
        loss = criterion(outputs.squeeze(1), y_batch.squeeze()) 
        loss.backward()
        optimizer.step()
print("Embedding learning complete. 🎉\n")

# ---------------------------------------------
# 2. Extract and Apply Embeddings (Transformation)
# ---------------------------------------------

def extract_and_transform(x_cat_np, x_num_np, model, embed_dims, total_emb_dim):
    """Extracts weights and transforms categorical data into a continuous array."""
    
    # Ensure model is in evaluation mode
    model.eval()
    
    embedding_matrices = [layer.weight.data.cpu().numpy() for layer in model.embedding_layers]
    N_samples = x_cat_np.shape[0]
    data_cat_emb = np.zeros((N_samples, total_emb_dim), dtype=np.float32)
    
    current_col = 0
    for i in range(x_cat_np.shape[1]):
        emb_matrix = embedding_matrices[i]
        emb_dim = embed_dims[i]
        
        # x_cat_np[:, i] contains the integer indices
        embedded_vectors = emb_matrix[x_cat_np[:, i]]
        
        data_cat_emb[:, current_col : current_col + emb_dim] = embedded_vectors
        current_col += emb_dim

    # Concatenate the new continuous categorical features with the numerical features
    data_continuous = np.hstack([x_num_np.astype(np.float32), data_cat_emb])
    
    return data_continuous, embedding_matrices

# Transform all three datasets
X_train_cont, embedding_matrices = extract_and_transform(x_cat_train_np, x_num_train_np, model, embed_dims, total_emb_dim)
X_val_cont, _ = extract_and_transform(x_cat_val_np, x_num_val_np, model, embed_dims, total_emb_dim)
X_test_cont, _ = extract_and_transform(x_cat_test_np, x_num_test_np, model, embed_dims, total_emb_dim)


# --- VISUALIZATION: AFTER EMBEDDING ---
print("="*90)
print("VISUALIZATION: X_train AFTER EMBEDDING (First 5 Rows)")

emb_cols = [f'Emb_{i}_{j}' for i in range(C_cat) for j in range(embed_dims[0])]
num_cols = [f'Num_{i}' for i in range(C_num)]

df_continuous_after = pd.DataFrame(X_train_cont[:5], columns=num_cols + emb_cols)

print(df_continuous_after.to_string(float_format='%.4f'))
print(f"Total columns AFTER embedding: {total_emb_dim} (Emb) + {C_num} (Num) = {X_train_cont.shape[1]}")
print("="*90)


# ---------------------------------------------
# 3. Saving the Fully Continuous Data Analogue, Embedding Matrices and Country Encoder
# ---------------------------------------------

print("\nSaving Embedding Matrices and Country_Encoder to:", OUTPUT_DIR)

# Save embedding matrices and country encoder
with open(os.path.join(OUTPUT_DIR, 'embedding_matrices.pkl'), 'wb') as f:
    pickle.dump(embedding_matrices, f)

with open(os.path.join(OUTPUT_DIR, 'country_encoder.pkl'), 'wb') as f:
    pickle.dump(country_encoder, f)

print("\nSaving fully continuous data to:", OUTPUT_DIR)

# Save the concatenated continuous features
np.save(os.path.join(OUTPUT_DIR, 'X_num_train.npy'), X_train_cont)
np.save(os.path.join(OUTPUT_DIR, 'X_num_val.npy'), X_val_cont)
np.save(os.path.join(OUTPUT_DIR, 'X_num_test.npy'), X_test_cont)

# Saving y (target)
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train_np)
np.save(os.path.join(OUTPUT_DIR, 'y_val.npy'), y_val_np)
np.save(os.path.join(OUTPUT_DIR, 'y_test.npy'), y_test_np)

print("All continuous files and empty categorical files have been saved in the 'churn2cont' directory.")

Loading data from specified paths...
Converting categorical strings to integer IDs...
Conversion complete. Categorical data is ready for embedding. ✅

VISUALIZATION: X_train BEFORE EMBEDDING (First 5 Rows - Encoded Integers + Floats)
     Num_0   Num_1   Num_2       Num_3  Num_4       Num_5       Num_6  Cat_0  Cat_1  Cat_2  Cat_3
0 838.0000 35.0000  0.0000      0.0000 2.0000 197305.9062 197305.9062      2      1      0      1
1 661.0000 44.0000  1.0000 141136.6250 1.0000 189742.7812 189742.7812      1      1      1      0
2 547.0000 62.0000 10.0000 127738.7500 2.0000  85153.0000  85153.0000      0      1      1      1
3 615.0000 41.0000  4.0000 130385.8203 1.0000 130661.9531 130661.9531      0      1      0      1
4 721.0000 40.0000  3.0000      0.0000 1.0000 144874.6719 144874.6719      0      1      1      1
Total columns BEFORE embedding: 7 (Num) + 4 (Cat) = 11

Training model to learn embeddings (Epochs: 5, Total Embedding Dim: 12)...
Embedding learning complete. 🎉

VISUALIZATION: 

### Decoder V1.1

### Revert Embedding

In [22]:
import numpy as np
import torch
import torch.nn as nn
from scipy.spatial.distance import cdist # Required for Nearest Neighbor search
import os
import pandas as pd

# --- Define Paths and Feature Counts ---
ENCODER_DIR = '/home/fjunpop/tab-ddpm-dp/data/churn2cont/' 
# SYNTHETICS_DIR = '/home/fjunpop/tab-ddpm-dp/exp/churn2lat/ddpm_cb_best'
# SYNTHETICS_DIR = '/home/fjunpop/tab-ddpm-dp/exp/churn2cont/churn2cont_dp_eps_100_best'
# SYNTHETICS_DIR = '/home/fjunpop/tab-ddpm-dp/exp/churn2cont/churn2cont_dp_eps_100_best'
SYNTHETICS_DIR = '/home/fjunpop/tab-ddpm-dp/exp/churn2cont/churn2cont_dp_eps_10_best'
# SYNTHETICS_DIR = '/home/fjunpop/tab-ddpm-dp/exp/churn2cont/churn2cont_dp_eps_1_best'
OUTPUT_DIR = os.path.join(SYNTHETICS_DIR, 'reverted')
os.makedirs(OUTPUT_DIR, exist_ok=True)

C_cat = 4 # 4 categorical features
C_num = 7 # 7 numerical features

# ---------------------------------------------
# 0. Load Saved Encoder Artifacts
# ---------------------------------------------
print("Loading encoder artifacts from disk...")

with open(os.path.join(ENCODER_DIR, 'embedding_matrices.pkl'), 'rb') as f:
    embedding_matrices = pickle.load(f)

with open(os.path.join(ENCODER_DIR, 'country_encoder.pkl'), 'rb') as f:
    country_encoder = pickle.load(f)

print("Encoder artifacts loaded. ✅")

# ---------------------------------------------
# 1. Load Synthetic Data (Continuous)
# ---------------------------------------------
x_num_train_synthetic = np.load(os.path.join(SYNTHETICS_DIR, 'X_num_train.npy'), allow_pickle=True)
y_train_synthetic = np.load(os.path.join(SYNTHETICS_DIR, 'y_train.npy'), allow_pickle=True)
print(f"Loaded synthetic continuous features. Shape: {x_num_train_synthetic.shape}")


# ---------------------------------------------
# 2. De-Embedding Function
# ---------------------------------------------

def revert_embedding(X_gen_cont, learned_embedding_matrices, original_num_cols):
    """
    Splits the continuous data and maps the embedded parts back to the closest integer ID 
    using the provided learned embedding matrices (Nearest Neighbor search).
    
    Returns: X_cat_gen (reverted integer categorical), X_num_gen (synthetic numerical)
    """
    
    # 1. Split the data
    X_num_gen = X_gen_cont[:, :original_num_cols]
    X_cat_emb_gen = X_gen_cont[:, original_num_cols:] 
    
    X_cat_gen_reverted = []
    current_col = 0
    
    # 2. Revert each categorical feature
    for emb_matrix in learned_embedding_matrices:
        emb_dim = emb_matrix.shape[1]
        
        # Select the columns corresponding to the current feature
        feature_emb_vectors = X_cat_emb_gen[:, current_col : current_col + emb_dim]
        current_col += emb_dim
        
        # Find the closest original embedding vector (Nearest Neighbor search)
        distances = cdist(feature_emb_vectors, emb_matrix, metric='euclidean')
        
        # The index of the minimum distance is the closest integer ID
        closest_ids = np.argmin(distances, axis=1)
        
        X_cat_gen_reverted.append(closest_ids.reshape(-1, 1))

    # 3. Stack the reverted categorical features (integer IDs)
    X_cat_gen = np.hstack(X_cat_gen_reverted)
    return X_cat_gen, X_num_gen

# --- Execute De-Embedding ---
print("\nExecuting de-embedding on synthetic data...")
# *** USING THE EXACT USER VARIABLE NAME HERE ***
X_cat_gen_reverted, X_num_gen = revert_embedding(x_num_train_synthetic, embedding_matrices, C_num)

print(f"Reverted Categorical Data Shape: {X_cat_gen_reverted.shape}")
print(f"Reverted Numerical Data Shape: {X_num_gen.shape}")

# -------------------------------------------------------------
# 3. Final Reconstruction (String Conversion)
# -------------------------------------------------------------
print("\n--- Final Reconstruction: Converting IDs to Original Strings ---")
# NOTE: The 'country_encoder' object is assumed to be defined in the notebook's scope.

# 1. Initialize the final categorical array with string type
X_cat_gen_reconstructed = X_cat_gen_reverted.copy().astype('U20') 

# 2. Feature 0: Country/Area (Requires OrdinalEncoder.inverse_transform)
# Reshape is crucial as inverse_transform expects a 2D array.
X_cat_gen_reconstructed[:, 0:1] = country_encoder.inverse_transform(X_cat_gen_reverted[:, 0:1].reshape(-1, 1))

# 3. Features 1, 2, 3: Binary '0'/'1' (Direct conversion from int 0/1 to str '0'/'1')
for i in range(1, C_cat):
    # Map the integer array (0, 1) to the string array ('0', '1')
    X_cat_gen_reconstructed[:, i] = X_cat_gen_reverted[:, i].astype(str)
    
print("Reconstruction complete. Final categorical data is back to original string types. ✅")

# ---------------------------------------------
# 4. Save Final Reconstructed Data
# ---------------------------------------------

print("\nSaving reverted data...")

# Save the two separated components of the synthetic dataset
np.save(os.path.join(OUTPUT_DIR, 'X_cat_train.npy'), X_cat_gen_reconstructed)
np.save(os.path.join(OUTPUT_DIR, 'X_num_train.npy'), X_num_gen)

# Save the synthetic labels (using the exact user variable name)
# *** USING THE EXACT USER VARIABLE NAME HERE ***
np.save(os.path.join(OUTPUT_DIR, 'y_train.npy'), y_train_synthetic)

print(f"Reverted categorical and numerical data saved to {OUTPUT_DIR} 🎉")

# --- Optional: Display Sample Reverted Data ---
print("\n" + "="*80)
print("SAMPLE REVERTED DATA (First 5 Rows)")
df_cat = pd.DataFrame(X_cat_gen_reverted[:5], columns=[f'Cat_{i}' for i in range(C_cat)])
df_num = pd.DataFrame(X_num_gen[:5], columns=[f'Num_{i}' for i in range(C_num)])
df_reverted_sample = pd.concat([df_num, df_cat], axis=1)
print(df_reverted_sample.to_string(float_format='%.4f'))
print("="*80)

Loading encoder artifacts from disk...
Encoder artifacts loaded. ✅
Loaded synthetic continuous features. Shape: (21600, 19)

Executing de-embedding on synthetic data...
Reverted Categorical Data Shape: (21600, 4)
Reverted Numerical Data Shape: (21600, 7)

--- Final Reconstruction: Converting IDs to Original Strings ---
Reconstruction complete. Final categorical data is back to original string types. ✅

Saving reverted data...
Reverted categorical and numerical data saved to /home/fjunpop/tab-ddpm-dp/exp/churn2cont/churn2cont_dp_eps_10_best/reverted 🎉

SAMPLE REVERTED DATA (First 5 Rows)
     Num_0   Num_1   Num_2       Num_3  Num_4       Num_5       Num_6  Cat_0  Cat_1  Cat_2  Cat_3
0 350.0120 18.0000 10.0000 250898.0938 1.0000     11.5800     11.5800      0      0      1      1
1 350.0000 18.0000  0.0000      0.0000 4.0000     11.5800     11.5800      1      0      0      1
2 850.0000 92.0000 10.0000      0.0000 1.0000     11.5800     11.5800      1      1      1      1
3 350.0000 18.

In [13]:
# Get unique values

# 1. Unique values across all categorical features in the training set
print("--- Unique values in x_cat_train (across all columns) ---")
unique_x_cat_train = np.unique(X_cat_gen_reconstructed)
print(unique_x_cat_train)

print("\n--- Unique values per categorical feature (column) in x_cat_train ---")
# Iterate through each column (feature) of the 2D array
for i in range(X_cat_gen_reconstructed.shape[1]):
    # Slice the array to get all rows for the i-th column: x_cat_train[:, i]
    unique_col_i = np.unique(X_cat_gen_reconstructed[:, i])
    print(f"Feature {i} (Cat_{i}): {len(unique_col_i)} unique IDs -> {unique_col_i}")


--- Unique values in x_cat_train (across all columns) ---
['0' '1' 'France' 'Germany' 'Spain']

--- Unique values per categorical feature (column) in x_cat_train ---
Feature 0 (Cat_0): 3 unique IDs -> ['France' 'Germany' 'Spain']
Feature 1 (Cat_1): 2 unique IDs -> ['0' '1']
Feature 2 (Cat_2): 2 unique IDs -> ['0' '1']
Feature 3 (Cat_3): 2 unique IDs -> ['0' '1']
